# C. elegans — Attribution and Feature Sensitivity Notebook: fastC-GC

Prerequisites: identity-preprocessed raw windows and the dataset `shared/recording` checkpoint from preprocessing. This notebook fits a separate purged chronological outer-owned feature bundle.
This notebook reports predictor-feature sensitivity only; it does not identify a biological intervention.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


Notebook interpreter: /Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/bin/python
Installed backends: {'fastcgc': '0.1.0', 'tigramite': '5.2.10.1', 'scikit-learn': '1.9.0', 'kmedoids': '0.5.5', 'igraph': '1.0.0', 'leidenalg': '0.12.0'}


In [2]:
from omegaconf import OmegaConf

from effectome.connectivity import ConnectivityConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_candidate_drivers,
    run_feature_sensitivity,
    run_owned_feature_artifacts,
)

DATASET_ID = "c_elegans"
DATASET_OUTPUT_ID = "bundle-net-c-elegans"
RECORDING_ID = "bundle-net-worm-0"
RUN_ID = "reference"
METHOD = "cgc"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RECORDING_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "recording" / "artifact.pkl"
WINDOWS_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "windows" / "artifact.pkl"
BEHAVIOR_KEY = "motif"
FIT_SCOPE_ID = f"{RUN_ID}/{METHOD}/sensitivity-outer-0"
MULTIPLICITY_FAMILY = (f"{BEHAVIOR_KEY}:feature-edit",)
connectivity_data = OmegaConf.to_container(
    OmegaConf.load(PROJECT_ROOT / "conf" / "connectivity" / f"{METHOD}.yaml"), resolve=True
)
CONNECTIVITY_CFG = ConnectivityConfig(**connectivity_data)
FORCE = False

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} attribution and feature sensitivity",
    expected_stages=[
        "recording_input",
        "outer_owned_features",
        "candidate_drivers",
        "surrogate_model",
        "feature_sensitivity",
    ],
    force=FORCE,
)


/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cgc attribution and feature sensitivity
run_id=reference method=cgc
run_dir=/Users/sadiqadedayo/Documents/projects/effectome-to-behavior/effectome-to-behavior/outputs/analysis/bundle-net-c-elegans/bundle-net-worm-0/reference/cgc
resume mode: FORCE=False, matching completed checkpoints will be reused
Rerun interrupted notebooks from the top; completed stages resume from checkpoints.
Stage status for reference/cgc:
  - candidate_drivers: not started
  - community: completed (signature=5e628d8394, artifact=yes)
  - connectivity: completed (signature=867a8d53f5, artifact=yes)
  - connectivity.chunk-00000: completed (signature=1b91e06b3b, artifact=yes)
  - connectivity.chunk-00001: completed (signature=1c728c3716, artifact=yes)
  - connectivity.chunk-00002: completed (signature=3b9a51a36a, artifact=yes)
  - connectivity.chunk-00003: completed (signature=fc4c5d2507, artifact=yes)
  - connectivity.chunk-00004: completed (signature=5d38173532, artifact=yes)
  - connectivity.chunk-00005: comple

In [3]:
owned = run_owned_feature_artifacts(
    run, windows_path=WINDOWS_PATH, connectivity_cfg=CONNECTIVITY_CFG,
    fit_scope_id=FIT_SCOPE_ID, evaluation_fraction=0.2, force=FORCE,
)
print(f"Training windows: {len(owned.fit_indices)}, held-out windows: {len(owned.evaluation_indices)}")

candidates = run_candidate_drivers(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=1,
    n_folds=5,
    embargo=4,
    seed=42,
    matched_control_percentile=95.0,
    fit_indices=owned.fit_indices,
    evaluation_indices=owned.evaluation_indices,
    fit_scope_id=owned.fit_scope_id,
    force=FORCE,
)
top = candidates.scores[0]
print(top)
print_stage_status(run)


[reference/cgc] windows_input: checking checkpoint; computing if needed...
[reference/cgc] windows_input: reused checkpoint in 0.5s
[reference/cgc] outer_owned_features: checking checkpoint; computing if needed...
[reference/cgc] outer_owned_features: completed in 1541.5s
Training windows: 125, held-out windows: 38
[reference/cgc] recording_input: checking checkpoint; computing if needed...
[reference/cgc] recording_input: reused checkpoint in 0.0s
[reference/cgc] candidate_drivers: checking checkpoint; computing if needed...


KeyboardInterrupt: 

In [ ]:
sensitivity = run_feature_sensitivity(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=1,
    ridge_alpha=1.0,
    n_folds=5,
    embargo=4,
    min_skill=0.0,
    scale=0.0,
    dose_scales=[1.0, 0.5, 0.0],
    top_k=1,
    endpoint="manifold",
    seed=42,
    fit_indices=owned.fit_indices,
    evaluation_indices=owned.evaluation_indices,
    fit_scope_id=owned.fit_scope_id,
    multiplicity_family=MULTIPLICITY_FAMILY,
    hypothesis_id=f"{BEHAVIOR_KEY}:feature-edit",
    min_absolute_skill=0.0,
    max_distribution_shift=5.0,
    n_controls=64,
    control_resolution_policy="unavailable",
    force=FORCE,
)
print(sensitivity)
print(sensitivity.analysis_kind)
print(sensitivity.biological_intervention_identified)
print_stage_status(run)
